# 03 · Features de ventana y etiqueta (exploración)

Ventanas de 14 y 30 días que solo miran hacia atrás, tendencias (últimos 30 días contra los 30 anteriores) y la etiqueta:
`target = 1` si el evento ocurre dentro de los próximos 1 a 30 días. Para los fallados se descarta todo lo posterior al
evento; para los sanos, sus últimos 30 días (censura: podrían fallar justo después de que terminan los datos).

**Entrada:** `salidas/fic.duckdb` · **Salida:** `salidas/dataset.parquet` (207.921 filas, 5.719 positivas).

In [ ]:
import os
import duckdb
import pandas as pd

# Rutas relativas a notebooks/exploracion/ (los CSV de Ford quedan en Datasets/, no versionado)
DATOS   = "../../Datasets/"
STATIC  = DATOS + "Static/"
TRIPS   = DATOS + "TripSummary/"
DYN     = DATOS + "Dynamic/"
SALIDAS = "salidas/"          # fic.duckdb, dataset.parquet, etc. (no versionado)
os.makedirs(SALIDAS, exist_ok=True)

# DuckDB siempre con límite de recursos (regla del proyecto: sin límite puede congelar una máquina de 16 GB)
con = duckdb.connect(SALIDAS + "fic.duckdb", config={"memory_limit": "4GB", "threads": 4})
con.sql("SET TimeZone='UTC'")

In [13]:
con.sql("""
create or replace table features as
select VehicleCode, dia, failed, Engine, ModelSeries, pais, ciudad, edad_dias, dias_hasta_evento,
       -- uso
       count(*)                over w30 as dias_activos_30d,
       sum(km)                 over w30 as km_30d,
       sum(viajes)             over w30 as viajes_30d,
       avg(pct_cortos)         over w30 as pct_cortos_30d,
       avg(pct_motor_frio)     over w30 as pct_motor_frio_30d,
       avg(vel_media)          over w30 as vel_media_30d,
       avg(temp_aire)          over w30 as temp_aire_30d,
       sum(min_ralenti)        over w30 as min_ralenti_30d,
       sum(ralentis_largos)    over w30 as ralentis_largos_30d,
       -- eficiencia y aceite
       avg(consumo_pct_100km)  over w30 as consumo_30d,
       min(aceite)             over w30 as aceite_30d,
       -- estado del filtro
       avg(pct_lleno)          over w14 as pct_lleno_14d,
       avg(pct_lleno)          over w30 as pct_lleno_30d,
       avg(acum_media)         over w30 as acum_media_30d,
       max(acum_max)           over w30 as acum_max_30d,
       sum(viajes_lleno)       over w30 as viajes_lleno_30d,
       sum(regeneraciones)     over w30 as regeneraciones_30d,
       avg(dist_regen)         over w30 as dist_regen_30d,
       sum(limpiezas_cortadas) over w30 as limpiezas_cortadas_30d,
       -- mecanismo de regeneración
       sum(viajes_filtro_cargado) over w30 as viajes_filtro_cargado_30d,
       sum(cortos_filtro_cargado) over w30 as cortos_filtro_cargado_30d,
       sum(regen_interrumpidas)   over w30 as regen_interrumpidas_30d,
       sum(regen_completas)       over w30 as regen_completas_30d,
       sum(regen_completas) over w30 / nullif(sum(regen_completas) over w30
           + sum(regen_interrumpidas) over w30 + sum(cortos_filtro_cargado) over w30, 0) as tasa_regen_exitosa_30d,
       -- tendencias: últimos 30 días menos los 30 anteriores
       avg(pct_lleno)         over w30 - avg(pct_lleno)         over w_prev as tend_lleno,
       avg(consumo_pct_100km) over w30 - avg(consumo_pct_100km) over w_prev as tend_consumo,
       avg(dist_regen)        over w30 - avg(dist_regen)        over w_prev as tend_dist_regen
from diario
window w14    as (partition by VehicleCode order by dia
                  range between interval 13 days preceding and current row),
       w30    as (partition by VehicleCode order by dia
                  range between interval 29 days preceding and current row),
       w_prev as (partition by VehicleCode order by dia
                  range between interval 59 days preceding and interval 30 days preceding)
""")

In [14]:
con.sql("""
create or replace table dataset as
select f.* exclude (ultimo_dia),
       case when failed = 1 and dias_hasta_evento between 1 and 30 then 1 else 0 end as target
from (select *, max(dia) over (partition by VehicleCode) as ultimo_dia from features) f
where (failed = 1 and dias_hasta_evento >= 1)
   or (failed = 0 and dia <= ultimo_dia - 30)
""")

con.sql("""
select failed, target, count(*) as filas, count(distinct VehicleCode) as vehiculos
from dataset group by all order by 1, 2
""").df()

,failed,target,filas,vehiculos
0,0,0,148671,694
1,1,0,53531,275
2,1,1,5719,271


In [15]:
con.sql("""
select target,
       round(avg(pct_lleno_30d), 4)          as lleno,
       round(avg(tend_lleno), 4)             as tendencia_lleno,
       round(avg(consumo_30d), 2)            as consumo,
       round(avg(pct_cortos_30d), 3)         as cortos,
       round(avg(dist_regen_30d), 0)         as dist_regen,
       round(avg(limpiezas_cortadas_30d), 3) as limpiezas_cortadas
from dataset
where failed = 1
group by target
""").df()

,target,lleno,tendencia_lleno,consumo,cortos,dist_regen,limpiezas_cortadas
0,0,0.0310,0.0025,17.13,0.462,325.0,0.019
1,1,0.0455,0.0069,17.81,0.455,383.0,0.022


In [16]:
con.sql(f"copy dataset to '{SALIDAS}dataset.parquet' (format parquet)")

df = pd.read_parquet(SALIDAS + "dataset.parquet")
print(df.shape)
df.isna().mean().sort_values(ascending=False).head(10)

(207921, 37)


dias_hasta_evento         0.715036
tend_dist_regen           0.357044
dist_regen_30d            0.240404
tend_consumo              0.115592
tasa_regen_exitosa_30d    0.112822
tend_lleno                0.100355
ciudad                    0.025034
aceite_30d                0.009403
viajes_lleno_30d          0.009244
vel_media_30d             0.007556
dtype: float64

In [17]:
con.close()
